In [80]:
import os, time, math as dt
from datetime import datetime
import dotenv
import geopandas as gpd
import pandas as pd
import numpy as np
import logging
import gc 
from glob import glob
import xlsxwriter
import openpyxl
import matplotlib.pyplot as plt
from io import BytesIO
import re
import matplotlib


gc.enable()

dotenv.load_dotenv()


wrk_dir=os.getenv("WORKING_DIR")
brwmb_targets_csv=os.getenv("BRWMB_TARGETS")
source_data=os.getenv("source_data")
output_parquets=os.getenv("OUTPUT_PARQUETS")
marxan_inputs=os.getenv("MARXAN_INPUTS")
input_parquets=os.getenv("INPUT_PARQUETS")
output_dir = os.path.join(wrk_dir, "brwmb_parquets")
mineral_licks_gdb=os.path.join(marxan_inputs,'MineralLicks_Hex_2024_12_16','MineralLicks_Hex_2024_12_16.gdb')
licks_layer='MineralLicks_2024_12_16'
grid_path=os.path.join(input_parquets,'hexagons_3ha.parquet')
grid_id_col='GRID_ID'
cia1_path=os.path.join(input_parquets,'cia_1.parquet')
cia2_path=os.path.join(input_parquets,'cia_2.parquet')
weighted_surface_path = os.path.join(output_dir, "step_3f_weighted_surface.parquet")
scenario5_setup_gdb=os.path.join(source_data,'Scenario5_Setup.gdb')
wmb_layer='priority_WMB'
rec_for_path=os.path.join(output_dir,'step_3c_Recruitment_Forest.parquet')
aflb_parquet=os.path.join(input_parquets,'thlb_select.parquet')
step2_lyr=os.path.join(output_parquets,'ret_class_2e_all.parquet')
target_crs = "EPSG:3005"
wmv_inputs=os.getenv("WMV_INPUTS")
ret_class=os.path.join(input_parquets,'rec_class_all.parquet')
table_outputs=os.getenv('TABLE_OUPUTS')
#set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
info=logging.info
debug=logging.debug
error=logging.error

dt_today=datetime.today().strftime('%Y%m%d')
out_path = os.path.join(table_outputs,f"scenario_5_summary_{dt_today}.xlsx")

input_data_gdb=os.path.join(source_data,'scen_5_inputs.gdb')
consrv_lnd_path=os.path.join(output_parquets,'step_3o_conservation_lands.parquet')  
scndry_rsrv_path=os.path.join(output_dir,'step_3i_secondary_reserves.parquet')
aspa_def_path=os.path.join(output_dir,'step_3j_aspatial_deferred_old_forest.parquet')
step_2e_path=os.path.join(output_dir, "ret_class_2e_all.parquet")
eco_png_path=os.path.join(input_parquets,'PNG_Critical_infrastructure.parquet')

hv1_l='hv1_clip'

eco_png=gdb=os.path.join(source_data,'ECONOMIC_PNG_Critical_Infrastructure_3March2026.gdb')
eco_png_layer=''


In [ ]:
#create summary from all existing tables 

def sheet_name_safe(name: str) -> str:
    bad = set(':\\/?*[]')
    cleaned = ''.join('_' if c in bad else c for c in name)
    return cleaned[:31] if len(cleaned) > 31 else cleaned

xlsx_files=glob(os.path.join(table_outputs,'*'))
sorted_names=[]
for f in xlsx_files:
    f=f.split(r'/')[-1]
    sorted_names.append(f)
sorted_names   

with pd.ExcelWriter(out_path, engine="xlsxwriter") as writer:
    workbook  = writer.book

    for i, f in enumerate(sorted_names, start=1):
        info(f" worksheet{f}")
        fo=os.path.join(table_outputs,f)
        if f.endswith('xlsx'):
            df = pd.read_excel(fo)
        elif f.endswith('csv'):
             df = pd.read_csv(fo)
        
        pattern = r"cat_short"  # your regex
        matches = [c for c in df.columns if re.search(pattern, c, flags=re.IGNORECASE)]
        if not matches:
            info("[rename] No columns match regex 'cat_short' (case-insensitive).")
        else:
            old = matches[0]
            new = "For_Rep"
            if new in df.columns and new != old:
                info(f"[rename] Target name '{new}' already exists; not renaming '{old}'.")
            else:
                df = df.rename(columns={old: new})
                info(f"[rename] Renamed column '{old}' → '{new}'.")

        
        sheet = sheet_name_safe(f)
        df.to_excel(writer, index=False, sheet_name=sheet)
        worksheet = writer.sheets[sheet]
        (nrows, ncols) = df.shape
        # for col_idx, col in enumerate(df.columns):
        #             width = max(10, min(40, int(df[col].astype(str).str.len().clip(upper=40).mean()) + 2))
        #             worksheet.set_column(col_idx, col_idx, width)

        #add matplot lib image for each 
        # worksheet.insert_image(1, ncols + 1, f"{sheet}_plot.png", {
        #                 "image_data": img,
        #                 "x_scale": 1.0,
        #                 "y_scale": 1.0,
        #                 "object_position": 2,  # move but don't size with cells (optional)
        #             })




2026-03-09 11:18:22,874 - INFO -  worksheetscenario_5_summary.xlsx
2026-03-09 11:18:23,494 - INFO - [rename] No columns match regex 'cat_short' (case-insensitive).
2026-03-09 11:18:23,497 - INFO -  worksheetstep_2a_b_c_targets.xlsx
2026-03-09 11:18:24,024 - INFO - [rename] No columns match regex 'cat_short' (case-insensitive).
2026-03-09 11:18:24,029 - INFO -  worksheetstep_2d_summary_by_target.csv
2026-03-09 11:18:24,333 - INFO - [rename] No columns match regex 'cat_short' (case-insensitive).
2026-03-09 11:18:24,337 - INFO -  worksheetstep_3k_tsu.xlsx
2026-03-09 11:18:24,810 - INFO - [rename] No columns match regex 'cat_short' (case-insensitive).
2026-03-09 11:18:24,812 - INFO -  worksheetstep_3k_wmb.xlsx
2026-03-09 11:18:22,798 - INFO - [rename] No columns match regex 'cat_short' (case-insensitive).
2026-03-09 11:18:22,801 - INFO -  worksheetstep_3L_scndR_aspatiallyDef_for_rep.xlsx
2026-03-09 11:18:23,257 - INFO - [rename] Renamed column 'Rec_Cat_short' → 'For_Rep'.
2026-03-09 11:18:

In [102]:
hv1_gdf = gpd.read_file(input_data_gdb, layer=hv1_l)
tsu = gpd.read_file(input_data_gdb, layer='TSU')
wmb = gpd.read_file(input_data_gdb, layer='priority_WMB')
hv1_dz_gdf=hv1_gdf[hv1_gdf['Zone']== 'Development']

/home/cfolkers/miniforge3/envs/stac_tools/lib/python3.11/site-packages/pyogrio/raw.py:198: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiPolygon' is converted to 'MultiPolygon Z'
  return ogr_read(


In [105]:
consrv_lands=gpd.read_parquet(consrv_lnd_path)
scndry_rsrv=gpd.read_parquet(scndry_rsrv_path)
aspa_def=gpd.read_parquet(aspa_def_path)
aspa_polys = aspa_def[aspa_def.geometry.geom_type.isin(["Polygon", "MultiPolygon"])]
step_2e=gpd.read_parquet(step_2e_path)
eco_png=gpd.read_parquet(eco_png_path)

In [106]:
#step 1 and 2 from accounting
remove_from=pd.concat([hv1_dz_gdf, eco_png], ignore_index=True).dissolve()
scndry_rsrv_rmv=scndry_rsrv.overlay(remove_from, how='difference')
aspa_def_rmv=aspa_polys.overlay(remove_from, how='difference')
#find difference 
aspa_diss=aspa_polys.dissolve()
aspa_diss['tota_ha']=aspa_diss.geometry.area/10000
scndry_rsrv_dss=scndry_rsrv.dissolve()
scndry_rsrv_dss['tota_ha']=scndry_rsrv_dss.geometry.area/10000
scndry_rsrv_rmv_diss=scndry_rsrv_rmv.dissolve()
scndry_rsrv_rmv_diss['tota_ha']=scndry_rsrv_rmv_diss.geometry.area/10000
aspa_def_rmv_diss=aspa_def_rmv.dissolve()
aspa_def_rmv_diss['tota_ha']=aspa_def_rmv_diss.geometry.area/10000
data ={ 'Area':['Aspatially Deferred Old Forest', 'Secondary Reserves'],
       'Total Area (ha)':[aspa_diss['tota_ha'].values[0], scndry_rsrv_dss['tota_ha'].values[0]],
       'Area After HV1 DZ and PNG Removed (ha)':[aspa_def_rmv_diss['tota_ha'].values[0], scndry_rsrv_rmv_diss['tota_ha'].values[0]]
    #    'Percent Removed (%)':[round(aspa_def_rmv_diss['tota_ha'].values[0]/aspa_diss['tota_ha'].values[0]*100,2),
    #                           round(scndry_rsrv_rmv_diss['tota_ha'].values[0]/scndry_rsrv_dss['tota_ha'].values[0]*100,2)]
       }

summary_1_2=pd.DataFrame(data)
summary_1_2.to_excel(os.path.join(table_outputs,'accounting_step_1_2.xlsx')) 


In [ ]:
#step 3 
wmb = wmb.rename(columns={'aflb_25_percent':'wmb_aflb_25_percent', 'SUM_aflb_ha':'wmb_SUM_aflb_ha'})
tsu=tsu.rename(columns={'aflb_33_percent':'tsu_aflb_33_percent', 'SUM_aflb_ha':'tsu_SUM_aflb_ha'})
# step_2e_diss=step_2e.dissolve()
step2e_wmb=step_2e.sjoin(wmb, how='inner', predicate='intersects')
step2e_tsu=step_2e.sjoin(tsu, how='inner', predicate='intersects')

step2e_wmb['total_ha']=step2e_wmb.geometry.area/10000
step2e_wmb['aflb_area_ha']=step2e_wmb['aflb_fact']*step2e_wmb['total_ha']
step2e_tsu['total_ha']=step2e_tsu.geometry.area/10000
step2e_tsu['aflb_area_ha']=step2e_tsu['aflb_fact']*step2e_tsu['total_ha']

step2e_wmb_grouped=step2e_wmb.groupby('BASIN_ID_1', as_index=False).agg(
    step2e_aflb=('aflb_area_ha','sum'),
    step2e_total=('total_ha','sum')
)
step2e_wmb_grouped=step2e_wmb_grouped.rename(columns={'BASIN_ID_1':'BASIN_ID'})
step2e_tsu_grouped=step2e_tsu.groupby('TRAPLINE_AREA_IDENTIFIER_1', as_index=False).agg(
    step2e_aflb=('aflb_area_ha','sum'),
    step2e_total=('total_ha','sum')
)
step2e_tsu_grouped=step2e_tsu_grouped.rename(columns={'TRAPLINE_AREA_IDENTIFIER_1':'TRAPLINE_AREA_IDENTIFIER'})

wmb=wmb[['BASIN_ID', 'NAME','wmb_SUM_aflb_ha','wmb_aflb_25_percent', 'geometry']]
wmb['wmb_area_ha']=wmb.geometry.area/10000
tsu=tsu[['TRAPLINE_AREA_IDENTIFIER', 'tsu_SUM_aflb_ha','tsu_aflb_33_percent', 'geometry']]
tsu['tsu_area_ha']=tsu.geometry.area/10000

#join step2e_wmb_grouped on BASIN ID
step2e_wmb_joined=wmb.merge(step2e_wmb_grouped, on='BASIN_ID', how='left')
#join step2e_tsu_grouped on TRAPLINE_AREA_IDENTIFIER
step2e_tsu_joined=tsu.merge(step2e_tsu_grouped, on='TRAPLINE_AREA_IDENTIFIER', how='left')

wmb_order=['BASIN_ID', 'NAME', 'wmb_area_ha', 'wmb_SUM_aflb_ha', 'wmb_aflb_25_percent','step2e_total', 'step2e_aflb', 'geometry']
tsu_order=['TRAPLINE_AREA_IDENTIFIER', 'tsu_area_ha', 'tsu_SUM_aflb_ha','tsu_aflb_33_percent', 'step2e_total','step2e_aflb', 'geometry']
step2e_wmb_joined=step2e_wmb_joined[wmb_order]
step2e_tsu_joined=step2e_tsu_joined[tsu_order]

step2e_wmb_joined=step2e_wmb_joined.rename(columns={'wmb_area_ha':'WMB Total Area (ha)',
                                                    'wmb_SUM_aflb_ha':'WMB Total AFLB Area (ha)',
                                                'wmb_aflb_25_percent':'WMB AFLB 25 Percent (ha)',
                                                'step2e_aflb':'Step 2e Total AFLB Area (ha)',
                                                'step2e_total':'Step 2e Total Area (ha)'})
step2e_tsu_joined=step2e_tsu_joined.rename(columns={'tsu_area_ha':'TSU Total Area (ha)',
                                                    'tsu_SUM_aflb_ha':'TSU Total AFLB Area (ha)',
                                                'tsu_aflb_33_percent':'TSU AFLB 33 Percent (ha)',
                                                'step2e_aflb':'Step 2e Total AFLB Area (ha)',
                                                'step2e_total':'Step 2e Total Area (ha)'})

step2e_wmb_joined['Percent of 2e AFLB Target Achieved'] = (step2e_wmb_joined['Step 2e Total AFLB Area (ha)'] / step2e_wmb_joined['WMB Total AFLB Area (ha)'])*100
step2e_wmb_joined['Meets 25 Percent AFLB Target'] = step2e_wmb_joined['Step 2e Total AFLB Area (ha)'] >= step2e_wmb_joined['WMB AFLB 25 Percent (ha)']

step2e_tsu_joined['Percent of 2e AFLB Target Achieved'] = (step2e_tsu_joined['Step 2e Total AFLB Area (ha)'] / step2e_tsu_joined['TSU Total AFLB Area (ha)'])*100
step2e_tsu_joined['Meets 33 Percent AFLB Target'] = step2e_tsu_joined['Step 2e Total AFLB Area (ha)'] >= step2e_tsu_joined['TSU AFLB 33 Percent (ha)']

step2e_tsu_joined.to_excel(os.path.join(table_outputs,'accounting_step_3_tsu.xlsx'), index=False)
step2e_wmb_joined.to_excel(os.path.join(table_outputs,'accounting_step_3_wmb.xlsx'), index=False)


In [141]:
step2e_wmb['Rec_Cat_short']


0        MPD
1        MPD
2        MPD
3        MPD
4        MPD
        ... 
77613    MPM
77614    MPM
77615    MPM
77616    MPM
77617    MPM
Name: Rec_Cat_short, Length: 77946, dtype: object

In [145]:
wmb

,BASIN_ID,NAME,wmb_SUM_aflb_ha,wmb_aflb_25_percent,geometry,wmb_area_ha
0,195786,Lower Sikanni Chief River,237613.062285,59403.265571,"MULTIPOLYGON (((1290272.476 1448813.532, 12902...",309789.456792
1,195800,Blueberry River,175091.823230,43772.955807,"MULTIPOLYGON (((1258807.685 1354678.763, 12587...",296019.443572
2,195795,Upper Beatton River,281362.801454,70340.700363,"MULTIPOLYGON (((1290557.825 1402059.405, 12905...",339648.859100
3,195796,Middle Beatton River,57059.994669,14264.998667,"MULTIPOLYGON (((1292703.216 1373052.546, 12927...",89743.335970


In [143]:
#step 4

step2e_wmb_grouped=step2e_wmb.groupby(['BASIN_ID_1','Rec_Cat','Rec_Cat_short'], as_index=False).agg(
    step2e_aflb=('aflb_area_ha','sum'),
    step2e_total=('total_ha','sum'))

   BASIN_ID_1  Rec_Cat Rec_Cat_short  step2e_aflb  step2e_total
0      195800      1.0           LPC  8038.220716   8423.672668
1      195800      1.0           LPM   490.445507    493.961857
2      195800      1.0           MPC  1492.998108   1499.037131
3      195800      1.0           MPM   685.332041    685.348640
4      195800      2.0           HPC   694.338602    694.344788
..        ...      ...           ...          ...           ...
87     195800      3.0           MPD  1458.965915   1458.965915
88     195800      4.0           HPC  2197.066046   2197.066046
89     195800      4.0           HPD   213.336160    213.336160
90     195800      4.0           HPM   938.438041    938.438041
91     195800      4.0           LPD    72.667590     72.667590

[92 rows x 5 columns]
